# LLM Pokemon Tournament Analysis

Interactive analysis of round-robin tournament logs. This notebook provides:
1. Data loading and summary
2. Statistical analysis (win rates, confidence calibration, tool usage)
3. Exploration tools for investigating specific matches and decisions

## 1. Setup & Data Loading

In [ ]:
import json
import re
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional
import pandas as pd
from IPython.display import display, Markdown, HTML

# Configuration - update this path as needed
TOURNAMENT_PATH = Path("../../logs/round-robin-2026-01-27-1056")

In [ ]:
# Data structures

@dataclass
class Turn:
    """A single turn from a player's perspective."""
    turn_num: int
    observation: str
    state: str
    action: Optional[str]
    reasoning: Optional[str]
    prediction: Optional[str]
    confidence: Optional[int]  # 0-100
    tool_calls: list[dict]
    tokens: dict  # input, output, reasoning
    latency_ms: int
    raw_response: str

@dataclass  
class PlayerLog:
    """All turns from one player in a match."""
    player_id: str
    model: str
    team: str
    turns: list[Turn]

@dataclass
class ProtocolEvent:
    """A single event from the battle protocol."""
    turn: int
    event_type: str
    args: list

@dataclass
class Match:
    """A complete match with all data."""
    match_id: int
    battle_id: str
    model_a: str
    model_b: str
    team_a: str
    team_b: str
    winner: Optional[str]
    total_turns: int
    protocol: list[ProtocolEvent]
    player_logs: dict[str, PlayerLog]  # player_id -> log
    metadata: dict

@dataclass
class Tournament:
    """Complete tournament data."""
    name: str
    format: str
    models: list[str]
    games_per_pair: int
    matches: list[Match]
    manifest: dict

In [ ]:
# Parsing utilities

def extract_thinking(raw_response: str) -> Optional[str]:
    """Extract content from <thinking> blocks."""
    pattern = r'<thinking>(.*?)</thinking>'
    matches = re.findall(pattern, raw_response, re.DOTALL)
    return '\n---\n'.join(matches) if matches else None

def parse_confidence(conf_str: Optional[str]) -> Optional[int]:
    """Parse confidence string like '75%' or '** 75' to int."""
    if not conf_str:
        return None
    # Remove common prefixes like '** '
    cleaned = re.sub(r'^[\*\s]+', '', str(conf_str))
    # Extract digits
    match = re.search(r'(\d+)', cleaned)
    return int(match.group(1)) if match else None

def parse_player_log(path: Path, metadata: dict) -> PlayerLog:
    """Parse a player's JSONL log file."""
    player_id = path.stem
    player_info = metadata.get('players', {}).get(player_id, {})
    
    turns = []
    with open(path) as f:
        for line in f:
            if not line.strip():
                continue
            data = json.loads(line)
            parsed = data.get('parsed', {})
            
            turn = Turn(
                turn_num=data.get('turn', 0),
                observation=data.get('observation', ''),
                state=data.get('state', ''),
                action=parsed.get('action'),
                reasoning=extract_thinking(data.get('raw_response', '')),
                prediction=parsed.get('prediction'),
                confidence=parse_confidence(data.get('confidence')),
                tool_calls=data.get('tool_calls', []),
                tokens=data.get('tokens', {}),
                latency_ms=data.get('latency_ms', 0),
                raw_response=data.get('raw_response', '')
            )
            turns.append(turn)
    
    return PlayerLog(
        player_id=player_id,
        model=player_info.get('model', 'unknown'),
        team=player_info.get('team', 'unknown'),
        turns=turns
    )

def parse_protocol(path: Path) -> list[ProtocolEvent]:
    """Parse protocol.jsonl into structured events."""
    events = []
    with open(path) as f:
        for line in f:
            if not line.strip():
                continue
            data = json.loads(line)
            turn = data.get('turn', 0)
            for event in data.get('events', []):
                if len(event) >= 2:
                    events.append(ProtocolEvent(
                        turn=turn,
                        event_type=event[1] if event[1] else 'empty',
                        args=event[2:] if len(event) > 2 else []
                    ))
    return events

def load_match(match_dir: Path, manifest_match: dict) -> Match:
    """Load a complete match from its directory."""
    metadata_path = match_dir / 'metadata.json'
    protocol_path = match_dir / 'protocol.jsonl'
    
    with open(metadata_path) as f:
        metadata = json.load(f)
    
    protocol = parse_protocol(protocol_path) if protocol_path.exists() else []
    
    # Load player logs
    player_logs = {}
    for p in match_dir.glob('*.jsonl'):
        if p.name != 'protocol.jsonl':
            log = parse_player_log(p, metadata)
            player_logs[log.player_id] = log
    
    outcome = metadata.get('outcome', {})
    
    return Match(
        match_id=manifest_match['id'],
        battle_id=metadata.get('battle_id', ''),
        model_a=manifest_match['model_a'],
        model_b=manifest_match['model_b'],
        team_a=manifest_match['team_a'],
        team_b=manifest_match['team_b'],
        winner=manifest_match.get('winner'),
        total_turns=outcome.get('total_turns', 0),
        protocol=protocol,
        player_logs=player_logs,
        metadata=metadata
    )

def load_tournament(path: Path) -> Tournament:
    """Load complete tournament data."""
    manifest_path = path / 'manifest.json'
    with open(manifest_path) as f:
        manifest = json.load(f)
    
    tourney_info = manifest['tournament']
    matches = []
    
    for m in manifest['matches']:
        match_dir = path / f"match-{m['id']:04d}"
        if match_dir.exists():
            matches.append(load_match(match_dir, m))
    
    return Tournament(
        name=tourney_info['name'],
        format=tourney_info['format'],
        models=tourney_info['models'],
        games_per_pair=tourney_info['games_per_pair'],
        matches=matches,
        manifest=manifest
    )

In [ ]:
# Load the tournament
tournament = load_tournament(TOURNAMENT_PATH)
print(f"Loaded tournament: {tournament.name}")
print(f"Format: {tournament.format}")
print(f"Models: {', '.join(tournament.models)}")
print(f"Matches: {len(tournament.matches)}")

In [ ]:
# Match summary table
match_data = []
for m in tournament.matches:
    match_data.append({
        'Match': m.match_id,
        'Model A': m.model_a,
        'Model B': m.model_b,
        'Winner': m.winner or 'N/A',
        'Turns': m.total_turns,
        'Team A': m.team_a,
        'Team B': m.team_b
    })

df_matches = pd.DataFrame(match_data)
display(df_matches)

## 2. Statistical Analysis

In [ ]:
# Win rates
def compute_win_rates(tournament: Tournament) -> pd.DataFrame:
    """Compute overall and head-to-head win rates."""
    wins = {m: 0 for m in tournament.models}
    games = {m: 0 for m in tournament.models}
    
    for match in tournament.matches:
        games[match.model_a] += 1
        games[match.model_b] += 1
        if match.winner:
            wins[match.winner] += 1
    
    data = []
    for model in tournament.models:
        rate = wins[model] / games[model] * 100 if games[model] > 0 else 0
        data.append({
            'Model': model,
            'Wins': wins[model],
            'Games': games[model],
            'Win Rate': f"{rate:.1f}%"
        })
    
    return pd.DataFrame(data)

display(Markdown("### Overall Win Rates"))
display(compute_win_rates(tournament))

In [ ]:
# Head-to-head matrix
def head_to_head(tournament: Tournament) -> pd.DataFrame:
    """Create head-to-head win matrix."""
    models = tournament.models
    matrix = {m: {n: 0 for n in models} for m in models}
    
    for match in tournament.matches:
        if match.winner:
            loser = match.model_b if match.winner == match.model_a else match.model_a
            matrix[match.winner][loser] += 1
    
    df = pd.DataFrame(matrix).T
    df.index.name = 'Winner ↓ / Loser →'
    return df

display(Markdown("### Head-to-Head Wins"))
display(head_to_head(tournament))

In [ ]:
# Game length by model
def game_length_stats(tournament: Tournament) -> pd.DataFrame:
    """Average game length when model wins vs loses."""
    from collections import defaultdict
    
    win_turns = defaultdict(list)
    lose_turns = defaultdict(list)
    
    for match in tournament.matches:
        if match.winner:
            loser = match.model_b if match.winner == match.model_a else match.model_a
            win_turns[match.winner].append(match.total_turns)
            lose_turns[loser].append(match.total_turns)
    
    data = []
    for model in tournament.models:
        avg_win = sum(win_turns[model]) / len(win_turns[model]) if win_turns[model] else 0
        avg_lose = sum(lose_turns[model]) / len(lose_turns[model]) if lose_turns[model] else 0
        data.append({
            'Model': model,
            'Avg Turns (Winner)': f"{avg_win:.1f}" if win_turns[model] else 'N/A',
            'Avg Turns (Loser)': f"{avg_lose:.1f}" if lose_turns[model] else 'N/A'
        })
    
    return pd.DataFrame(data)

display(Markdown("### Game Length by Outcome"))
display(game_length_stats(tournament))

In [ ]:
# Confidence calibration over game phases
def confidence_by_phase(tournament: Tournament) -> pd.DataFrame:
    """Analyze confidence in early/mid/late game phases."""
    from collections import defaultdict
    
    # Collect confidence by model and phase
    phase_conf = defaultdict(lambda: {'early': [], 'mid': [], 'late': []})
    
    for match in tournament.matches:
        total_turns = match.total_turns
        for player_id, log in match.player_logs.items():
            # Determine model from player_id (e.g., 'DeepSeek-0' -> 'DeepSeek')
            model = log.model.split('/')[0] if '/' in log.model else log.model
            # Try to match against tournament models
            for m in tournament.models:
                if m.lower() in player_id.lower() or m.lower() in model.lower():
                    model = m
                    break
            
            for turn in log.turns:
                if turn.confidence is None:
                    continue
                # Determine phase
                if total_turns == 0:
                    phase = 'mid'
                elif turn.turn_num <= total_turns * 0.33:
                    phase = 'early'
                elif turn.turn_num <= total_turns * 0.66:
                    phase = 'mid'
                else:
                    phase = 'late'
                
                phase_conf[model][phase].append(turn.confidence)
    
    data = []
    for model in tournament.models:
        row = {'Model': model}
        for phase in ['early', 'mid', 'late']:
            vals = phase_conf[model][phase]
            avg = sum(vals) / len(vals) if vals else 0
            row[f'{phase.capitalize()} Game (avg conf)'] = f"{avg:.1f}%" if vals else 'N/A'
        data.append(row)
    
    return pd.DataFrame(data)

display(Markdown("### Confidence by Game Phase"))
display(confidence_by_phase(tournament))

In [ ]:
# Tool usage patterns
def tool_usage_stats(tournament: Tournament) -> pd.DataFrame:
    """Analyze which tools each model uses and how often."""
    from collections import defaultdict
    
    tool_counts = defaultdict(lambda: defaultdict(int))
    turn_counts = defaultdict(int)
    
    for match in tournament.matches:
        for player_id, log in match.player_logs.items():
            # Match model name
            model = None
            for m in tournament.models:
                if m.lower().replace('-', '') in player_id.lower().replace('-', ''):
                    model = m
                    break
            if not model:
                continue
            
            for turn in log.turns:
                turn_counts[model] += 1
                for tc in turn.tool_calls:
                    tool_name = tc.get('name', 'unknown')
                    tool_counts[model][tool_name] += 1
    
    # Get all tool names
    all_tools = set()
    for tools in tool_counts.values():
        all_tools.update(tools.keys())
    
    data = []
    for model in tournament.models:
        row = {'Model': model, 'Turns': turn_counts[model]}
        for tool in sorted(all_tools):
            count = tool_counts[model].get(tool, 0)
            per_turn = count / turn_counts[model] if turn_counts[model] > 0 else 0
            row[tool] = f"{count} ({per_turn:.2f}/turn)"
        data.append(row)
    
    return pd.DataFrame(data)

display(Markdown("### Tool Usage by Model"))
df_tools = tool_usage_stats(tournament)
display(df_tools)

In [ ]:
# Token efficiency
def token_stats(tournament: Tournament) -> pd.DataFrame:
    """Analyze token usage per model."""
    from collections import defaultdict
    
    tokens = defaultdict(lambda: {'input': [], 'output': [], 'reasoning': [], 'latency': []})
    
    for match in tournament.matches:
        for player_id, log in match.player_logs.items():
            model = None
            for m in tournament.models:
                if m.lower().replace('-', '') in player_id.lower().replace('-', ''):
                    model = m
                    break
            if not model:
                continue
            
            for turn in log.turns:
                t = turn.tokens
                if t.get('input'):
                    tokens[model]['input'].append(t['input'])
                if t.get('output'):
                    tokens[model]['output'].append(t['output'])
                if t.get('reasoning'):
                    tokens[model]['reasoning'].append(t['reasoning'])
                if turn.latency_ms:
                    tokens[model]['latency'].append(turn.latency_ms)
    
    data = []
    for model in tournament.models:
        t = tokens[model]
        data.append({
            'Model': model,
            'Avg Input Tokens': f"{sum(t['input'])/len(t['input']):.0f}" if t['input'] else 'N/A',
            'Avg Output Tokens': f"{sum(t['output'])/len(t['output']):.0f}" if t['output'] else 'N/A',
            'Avg Reasoning Tokens': f"{sum(t['reasoning'])/len(t['reasoning']):.0f}" if t['reasoning'] else 'N/A',
            'Avg Latency (ms)': f"{sum(t['latency'])/len(t['latency']):.0f}" if t['latency'] else 'N/A'
        })
    
    return pd.DataFrame(data)

display(Markdown("### Token Usage & Latency"))
display(token_stats(tournament))

## 3. Exploration Tools

In [ ]:
def get_match(match_id: int) -> Match:
    """Get a match by ID."""
    for m in tournament.matches:
        if m.match_id == match_id:
            return m
    raise ValueError(f"Match {match_id} not found")

def view_match(match_id: int):
    """Display match overview with key events."""
    m = get_match(match_id)
    
    print(f"=== Match {m.match_id}: {m.model_a} vs {m.model_b} ===")
    print(f"Winner: {m.winner or 'N/A'}")
    print(f"Total Turns: {m.total_turns}")
    print(f"Teams: {m.team_a} vs {m.team_b}")
    print()
    
    # Key events: faints, major damage
    print("Key Events:")
    for event in m.protocol:
        if event.event_type == 'faint':
            print(f"  Turn {event.turn}: FAINT - {event.args[0] if event.args else '?'}")
        elif event.event_type == '-immune':
            print(f"  Turn {event.turn}: IMMUNE - {event.args}")

In [ ]:
def view_turn(match_id: int, turn_num: int, player: Optional[str] = None):
    """Display detailed turn info for one or both players."""
    m = get_match(match_id)
    
    for player_id, log in m.player_logs.items():
        if player and player.lower() not in player_id.lower():
            continue
        
        for turn in log.turns:
            if turn.turn_num == turn_num:
                print(f"\n{'='*60}")
                print(f"Player: {player_id} ({log.model})")
                print(f"Turn: {turn_num}")
                print(f"{'='*60}")
                print(f"\n--- Observation ---\n{turn.observation}")
                print(f"\n--- Action ---\n{turn.action}")
                print(f"\n--- Prediction ---\n{turn.prediction}")
                print(f"\n--- Confidence ---\n{turn.confidence}%" if turn.confidence else "")
                print(f"\n--- Tool Calls ({len(turn.tool_calls)}) ---")
                for tc in turn.tool_calls:
                    print(f"  {tc.get('name', '?')}")
                print(f"\n--- Reasoning (excerpt) ---")
                if turn.reasoning:
                    # Show first 1000 chars
                    print(turn.reasoning[:1000] + ('...' if len(turn.reasoning) > 1000 else ''))

In [ ]:
def compare_reasoning(match_id: int, turn_num: int):
    """Side-by-side comparison of both players' reasoning for a turn."""
    m = get_match(match_id)
    
    print(f"=== Turn {turn_num} Comparison ===")
    print(f"Match {match_id}: {m.model_a} vs {m.model_b}\n")
    
    for player_id, log in m.player_logs.items():
        for turn in log.turns:
            if turn.turn_num == turn_num:
                print(f"\n{'─'*40}")
                print(f"🎮 {player_id}")
                print(f"Action: {turn.action}")
                print(f"Confidence: {turn.confidence}%" if turn.confidence else "")
                print(f"Prediction: {turn.prediction}")
                print(f"\nReasoning excerpt:")
                if turn.reasoning:
                    print(turn.reasoning[:800] + '...' if len(turn.reasoning) > 800 else turn.reasoning)

In [ ]:
def search_reasoning(query: str, max_results: int = 10):
    """Search for text in reasoning blocks."""
    results = []
    query_lower = query.lower()
    
    for match in tournament.matches:
        for player_id, log in match.player_logs.items():
            for turn in log.turns:
                if turn.reasoning and query_lower in turn.reasoning.lower():
                    results.append({
                        'match': match.match_id,
                        'turn': turn.turn_num,
                        'player': player_id,
                        'action': turn.action,
                        'excerpt': turn.reasoning[:200]
                    })
    
    print(f"Found {len(results)} results for '{query}':")
    for r in results[:max_results]:
        print(f"\n  Match {r['match']}, Turn {r['turn']} ({r['player']})")
        print(f"  Action: {r['action']}")
        print(f"  ...{r['excerpt']}...")

In [ ]:
def find_by_confidence(min_conf: int = 0, max_conf: int = 100):
    """Find turns with confidence in a specific range."""
    results = []
    
    for match in tournament.matches:
        for player_id, log in match.player_logs.items():
            for turn in log.turns:
                if turn.confidence is not None and min_conf <= turn.confidence <= max_conf:
                    results.append({
                        'match': match.match_id,
                        'turn': turn.turn_num,
                        'player': player_id,
                        'confidence': turn.confidence,
                        'action': turn.action,
                        'won': match.winner and player_id.lower().startswith(match.winner.lower().split('-')[0])
                    })
    
    df = pd.DataFrame(results)
    print(f"Found {len(results)} turns with confidence {min_conf}-{max_conf}%")
    return df

In [ ]:
def find_protocol_events(event_type: str):
    """Find all occurrences of a specific protocol event type."""
    results = []
    
    for match in tournament.matches:
        for event in match.protocol:
            if event.event_type == event_type:
                results.append({
                    'match': match.match_id,
                    'turn': event.turn,
                    'args': event.args,
                    'model_a': match.model_a,
                    'model_b': match.model_b
                })
    
    print(f"Found {len(results)} '{event_type}' events:")
    for r in results:
        print(f"  Match {r['match']}, Turn {r['turn']}: {r['args']}")
    return results

In [ ]:
def get_momentum(match_id: int) -> pd.DataFrame:
    """Track HP and faint differential over the course of a match."""
    m = get_match(match_id)
    
    # Track faints per side
    turns = []
    p1_faints = 0
    p2_faints = 0
    
    current_turn = 0
    for event in m.protocol:
        if event.turn != current_turn:
            turns.append({'turn': current_turn, 'p1_faints': p1_faints, 'p2_faints': p2_faints, 'diff': p2_faints - p1_faints})
            current_turn = event.turn
        
        if event.event_type == 'faint':
            if event.args and 'p1' in str(event.args[0]):
                p1_faints += 1
            elif event.args and 'p2' in str(event.args[0]):
                p2_faints += 1
    
    # Final state
    turns.append({'turn': current_turn, 'p1_faints': p1_faints, 'p2_faints': p2_faints, 'diff': p2_faints - p1_faints})
    
    return pd.DataFrame(turns)

## 4. Example Usage

In [ ]:
# Example: View match 1 overview
view_match(1)

In [ ]:
# Example: Find all moves that were immune
find_protocol_events('-immune')

In [ ]:
# Example: Search for mentions of "Levitate" in reasoning
search_reasoning('Levitate')

In [ ]:
# Example: Compare both players' reasoning on turn 1 of match 1
compare_reasoning(1, 1)